# 07 · DINO Attention Maps

Visualise the **last-layer CLS attention** from AstroDINO (ViT-B, patch=6, grid=12×12).

This is the same visualisation used in the DINO / DINOv2 papers: for each image,
the CLS token's attention weights to every patch in the **final transformer block**
are extracted and overlaid on the galaxy.  Different heads spontaneously specialise
in different spatial regions.

**Sections:**
1. Configuration
2. Dataset & Model
3. Attention Map Extraction
4. Per-head gallery (one example per class)
5. Mean-over-heads gallery (N examples per class)
6. Class-average attention maps

In [ ]:
import os, sys, glob
import numpy as np
import h5py
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from omegaconf import OmegaConf
from tqdm import tqdm
import matplotlib.pyplot as plt

PROJECT_ROOT = '/home/yacheng/ssl_outthere'
BENCH_ROOT   = os.path.join(PROJECT_ROOT, 'encoder_image/astrodino/benchmark')
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, BENCH_ROOT)

from dinov2.eval.setup import build_model_for_eval
from preprocessing import get_torgb

DEG_TO_PIXEL = 3600 * 1000 / 30
MORPH_NAMES  = {0: 'Spheroid', 1: 'Disk', 3: 'Bulge'}
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1 · Configuration

In [ ]:
MODEL_CONFIG  = f'{PROJECT_ROOT}/encoder_image/astrodino/model/astrodino_f150w_vitb_ps6_bs128/config.yaml'
MODEL_WEIGHTS = f'{PROJECT_ROOT}/encoder_image/astrodino/model/astrodino_f150w_vitb_ps6_bs128/eval/training_299999/teacher_checkpoint.pth'
DATA_ROOT     = f'{PROJECT_ROOT}/images/jwst/f150w'

REFF_MIN_PIX = 2.3
REFF_MAX_PIX = 72
N_PER_CLASS  = 50    # examples per class
N_SHOW       = 6     # gallery columns
SEED         = 42
rng = np.random.default_rng(SEED)

## 2 · Dataset & Model

In [ ]:
class MorphDataset(Dataset):
    def __init__(self, root, crop_size, reff_min, reff_max, n_per_class, seed=42):
        self.crop = transforms.CenterCrop(crop_size)
        _rng = np.random.default_rng(seed)
        self._files = []
        for fp in sorted(glob.glob(os.path.join(root, '*.h5'))):
            f = h5py.File(fp, 'r')
            if 'morph_flag_f150w' in f:
                self._files.append(f)
            else:
                f.close()
        self._idx = []
        for fi, f in enumerate(self._files):
            morph = f['morph_flag_f150w'][:]
            re    = f['radius_sersic'][:] * DEG_TO_PIXEL if 'radius_sersic' in f else None
            vmask = np.isfinite(morph) & (morph != 2)
            if re is not None:
                vmask &= np.isfinite(re) & (re >= reff_min) & (re <= reff_max)
            for li in np.where(vmask)[0]:
                self._idx.append((fi, li, int(morph[li])))
        by_cls = {}
        for i, (_, _, lbl) in enumerate(self._idx):
            by_cls.setdefault(lbl, []).append(i)
        n = min(n_per_class, min(len(v) for v in by_cls.values()))
        kept = []
        for lbl, idxs in sorted(by_cls.items()):
            chosen = _rng.choice(idxs, n, replace=False)
            kept.extend(chosen.tolist())
        self._idx = [self._idx[i] for i in kept]
        print(f'Dataset: {len(self._idx)} samples ({n}/class)')

    def __len__(self): return len(self._idx)

    def __getitem__(self, i):
        fi, li, lbl = self._idx[i]
        img = self._files[fi]['image'][li].astype('float32')
        img = np.repeat(img[np.newaxis], IN_CHANS if IN_CHANS > 1 else 1, axis=0)
        t   = self.crop(torch.from_numpy(img))
        t   = torch.from_numpy(TO_RGB(t.numpy()))
        return t, lbl

    def get_display_image(self, i):
        fi, li, _ = self._idx[i]
        img = self._files[fi]['image'][li].astype('float32')
        return self.crop(torch.from_numpy(img[np.newaxis])).squeeze(0).numpy()

    def close(self):
        for f in self._files:
            try: f.close()
            except: pass


cfg        = OmegaConf.load(MODEL_CONFIG)
model      = build_model_for_eval(cfg, pretrained_weights=MODEL_WEIGHTS)
model      = model.to(DEVICE).eval()
CROP_SIZE  = cfg.crops.global_crops_size
PATCH_SIZE = 6
N_PATCHES  = CROP_SIZE // PATCH_SIZE
TO_RGB, IN_CHANS = get_torgb(cfg)
N_REG = getattr(model, 'num_register_tokens', 0)
print(f'Model: crop={CROP_SIZE}  patch={PATCH_SIZE}  grid={N_PATCHES}x{N_PATCHES}')
print(f'n_heads={model.blocks[0][0].attn.num_heads}  N_REG={N_REG}')


In [ ]:

ds = MorphDataset(DATA_ROOT, CROP_SIZE, REFF_MIN_PIX, REFF_MAX_PIX, N_PER_CLASS, SEED)
classes = sorted({lbl for _, _, lbl in ds._idx})

# load all images + labels into memory (small dataset)
loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
img_list, lbl_list = [], []
for imgs, lbls in loader:
    img_list.append(imgs.cpu())
    lbl_list.append(np.array(lbls) if not isinstance(lbls, np.ndarray) else lbls)
ds.close()

img_all = torch.cat(img_list)                      # (N, C, H, W)
lbl_all = np.concatenate(lbl_list)                 # (N,)
img_np  = img_all.numpy()[:, 0]                    # (N, H, W) gray for display
print(f'Loaded: {img_all.shape}')

## 3 · Attention Map Extraction

`AttentionCapture` re-derives softmax attention weights from the QKV projection via a
forward hook.  This is version-agnostic (works with standard, xFormers, and SDPA paths).

In [ ]:
class AttentionCapture:
    """Forward hook: re-derives softmax attention from QKV of an Attention module."""
    def __init__(self):
        self.weights = None   # (B, n_heads, N_tok, N_tok)

    def __call__(self, module, inp, out):
        x = inp[0]
        B, N, D = x.shape
        nh = module.num_heads
        hd = D // nh
        with torch.no_grad():
            qkv = module.qkv(x).reshape(B, N, 3, nh, hd).permute(2, 0, 3, 1, 4)
            q, k = qkv[0], qkv[1]
            attn = (q @ k.transpose(-2, -1)) * (hd ** -0.5)
            self.weights = attn.softmax(dim=-1).cpu()   # (B, nh, N_tok, N_tok)


def get_last_block(mdl):
    return [sub for chunk in mdl.blocks for sub in chunk.children() if hasattr(sub, 'attn')][-1]


def get_cls_attention(mdl, imgs):
    """Returns (B, n_heads, N_PATCHES, N_PATCHES) CLS-to-patch attention from the last block."""
    cap = AttentionCapture()
    h   = get_last_block(mdl).attn.register_forward_hook(cap)
    with torch.no_grad():
        _ = mdl(imgs.to(DEVICE))
    h.remove()
    cls_to_patch = cap.weights[:, :, 0, 1 + N_REG:]   # skip CLS + register tokens
    return cls_to_patch.reshape(-1, cap.weights.shape[1], N_PATCHES, N_PATCHES)


def upsample(maps, size):
    """(B, H_p, W_p) -> (B, size, size) bilinear."""
    return F.interpolate(maps.unsqueeze(1).float(),
                         size=(size, size), mode='bilinear',
                         align_corners=False).squeeze(1)


n_heads = get_last_block(model).attn.num_heads
print(f'n_heads={n_heads}  N_PATCHES={N_PATCHES}  N_REG={N_REG}')

## 4 · Per-head Attention — One Example per Class

A ViT with 8 attention heads means that in each transformer block, the self-attention
is computed 8 times in parallel, each with its own learned Q/K/V projections.
Each head is free to learn a different "question": one might ask *where is the bright
core?*, another *where are the spiral arms?*, another *what is background sky?*

This is the visualisation from the original DINO paper (Caron et al. 2021): the authors
showed that, without any supervision, different heads spontaneously specialise into
semantically meaningful detectors.  For galaxies we expect similar behaviour — some
heads locking onto the nucleus, others onto extended structure.

Each column below is one head; each row is one morphology class.  The heatmap is the
CLS-to-patch attention weight (higher = the model "looked" more at that patch).

In [ ]:
N_EX = 3   # examples per class

sample_idx = [i for cls in classes
                for i in np.where(lbl_all == cls)[0][:N_EX].tolist()]
sample_t   = img_all[sample_idx]

attn = get_cls_attention(model, sample_t)   # (n_classes*N_EX, n_heads, G, G)

ext = [-0.5, CROP_SIZE - 0.5, -0.5, CROP_SIZE - 0.5]
fig, axes = plt.subplots(len(classes) * N_EX, n_heads + 1,
                         figsize=(2.2 * (n_heads + 1), 2.2 * len(classes) * N_EX))
for row, (cls_idx, img_idx) in enumerate(zip(
        [cls for cls in classes for _ in range(N_EX)], sample_idx)):
    axes[row, 0].imshow(img_np[img_idx], cmap='gray', origin='lower')
    axes[row, 0].axis('off')
    if row % N_EX == 0:
        axes[row, 0].set_ylabel(MORPH_NAMES[cls_idx], fontsize=11)
    for h in range(n_heads):
        ah = attn[row, h].numpy()
        ax = axes[row, h + 1]
        ax.imshow(img_np[img_idx], cmap='gray', origin='lower')
        ax.imshow(ah, cmap='hot', alpha=0.65, origin='lower',
                  extent=ext, interpolation='nearest')
        ax.axis('off')
        if row == 0:
            ax.set_title(f'Head {h}', fontsize=9)
axes[0, 0].set_title('Image', fontsize=9)
fig.suptitle('Last-layer CLS attention per head', fontsize=12)
plt.tight_layout()
plt.show()

## 5 · Mean-over-Heads Gallery

The per-head maps above are informative but noisy individually.  A simple summary is to
**average the 8 head maps** into a single attention map per image.  This gives a robust
picture of *overall* spatial attention — which patches the model collectively attended
to when forming the CLS embedding.

This is the version to use when you want one clean overlay per galaxy.  `N_SHOW`
correctly-classified examples are shown per morphology class.

In [ ]:
# collect N_SHOW examples per class
gal_idx  = np.concatenate([np.where(lbl_all == cls)[0][:N_SHOW] for cls in classes])
gal_lbls = lbl_all[gal_idx]

attn_gal = []
for i in range(0, len(gal_idx), 32):
    a = get_cls_attention(model, img_all[gal_idx[i:i+32]]).mean(dim=1)  # mean heads, (B, G, G)
    attn_gal.append(a)
attn_gal = torch.cat(attn_gal)   # (N_gal, G, G)

# Layout: each class gets 2 rows — top=galaxy images, bottom=attention maps
fig, axes = plt.subplots(len(classes) * 2, N_SHOW,
                         figsize=(2.6 * N_SHOW, 2.8 * len(classes) * 2))
for row, cls in enumerate(classes):
    r_img  = row * 2
    r_attn = row * 2 + 1
    sel = np.where(gal_lbls == cls)[0]
    for col, si in enumerate(sel[:N_SHOW]):
        # top row: galaxy image
        axes[r_img, col].imshow(img_np[gal_idx[si]], cmap='gray', origin='lower')
        axes[r_img, col].axis('off')

        # bottom row: attention map
        axes[r_attn, col].imshow(attn_gal[si].numpy(), cmap='hot', origin='lower')
        axes[r_attn, col].axis('off')

        if col == 0:
            axes[r_img,  col].set_ylabel(f'{MORPH_NAMES[cls]}\nimage',   fontsize=10)
            axes[r_attn, col].set_ylabel(f'{MORPH_NAMES[cls]}\nattn map', fontsize=10)
        if row == 0:
            axes[r_img, col].set_title(f'Ex {col+1}', fontsize=9)

fig.suptitle('Last-layer CLS attention (mean over heads)  —  top: galaxy  |  bottom: attention', fontsize=12)
plt.tight_layout()
plt.show()

## 6 · Class-Average Attention Maps

Average over all `N_PER_CLASS` examples — reveals the typical spatial structure the model attends to for each morphology.

In [ ]:
attn_all = []
for i in tqdm(range(0, len(img_all), 32), desc='Attention'):
    a = get_cls_attention(model, img_all[i:i+32]).mean(dim=1)  # (B, G, G)
    attn_all.append(a)
attn_all = torch.cat(attn_all)   # (N, G, G)

# Layout: 2 rows — top=mean galaxy image, bottom=mean attention map
fig, axes = plt.subplots(2, len(classes), figsize=(4.5 * len(classes), 4.2 * 2))
for col, cls in enumerate(classes):
    mask     = (lbl_all == cls)
    avg_attn = attn_all[mask].mean(0).numpy()
    avg_img  = img_np[mask].mean(0)

    # top: mean galaxy image
    axes[0, col].imshow(avg_img, cmap='gray', origin='lower')
    axes[0, col].set_title(MORPH_NAMES[cls], fontsize=13)
    axes[0, col].axis('off')
    if col == 0:
        axes[0, col].set_ylabel('Mean image', fontsize=11)

    # bottom: mean attention map
    im = axes[1, col].imshow(avg_attn, cmap='hot', origin='lower')
    axes[1, col].axis('off')
    plt.colorbar(im, ax=axes[1, col], fraction=0.046, pad=0.04)
    if col == 0:
        axes[1, col].set_ylabel('Mean attention', fontsize=11)

fig.suptitle(f'Mean CLS attention per class  (averaged over {N_PER_CLASS} examples)', fontsize=12)
plt.tight_layout()
plt.show()